# 07 - Advanced Analytics
Anomaly detection (Isolation Forest, LOF), clustering (DBSCAN, KMeans), PCA, permutation/SHAP explainability, climate-trend & heatwave analysis.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
%matplotlib inline


In [2]:
import pandas as pd
from src import config
from src.advanced_analytics import (detect_anomalies_isolation_forest, detect_anomalies_lof,
                                     cluster_dbscan, cluster_kmeans, run_pca,
                                     climate_trend_analysis, detect_heatwaves, country_ranking)

df = pd.read_csv(config.CLEAN_DATA_FILE, parse_dates=[config.DATETIME_COL])
cols = ['temperature_celsius','humidity','pressure_mb','wind_kph']

## Isolation Forest anomaly detection

In [3]:
if_labels, _ = detect_anomalies_isolation_forest(df, cols)
print('Anomalies flagged:', (if_labels == -1).sum())

2026-07-03 05:39:21 | src.advanced_analytics | INFO | Isolation Forest flagged 240 anomalies out of 12000 rows


Anomalies flagged: 240


## Local Outlier Factor

In [4]:
lof_labels = detect_anomalies_lof(df, cols)
print('LOF anomalies:', (lof_labels == -1).sum())

2026-07-03 05:39:22 | src.advanced_analytics | INFO | LOF flagged 240 anomalies out of 12000 rows


LOF anomalies: 240


## KMeans weather-regime clustering

In [5]:
k_labels, kmeans_model = cluster_kmeans(df, cols, n_clusters=4)
pd.Series(k_labels).value_counts().sort_index()

2026-07-03 05:39:22 | src.advanced_analytics | INFO | KMeans formed 4 clusters


0    2175
1    3615
2    2873
3    3337
Name: count, dtype: int64

## DBSCAN density-based clustering

In [6]:
db_labels = cluster_dbscan(df, cols)
pd.Series(db_labels).value_counts().sort_index()

2026-07-03 05:39:22 | src.advanced_analytics | INFO | DBSCAN found 1 clusters (+ noise points: 76)


-1       76
 0    11924
Name: count, dtype: int64

## PCA dimensionality reduction

In [7]:
pca_transformed, pca_model = run_pca(df, cols, n_components=2)
print('Explained variance ratio:', pca_model.explained_variance_ratio_)

2026-07-03 05:39:22 | src.advanced_analytics | INFO | PCA explained variance ratio: [0.366 0.251]


Explained variance ratio: [0.36636943 0.25081912]


## Climate trend analysis

In [8]:
trend = climate_trend_analysis(df, config.DATETIME_COL, 'temperature_celsius')
trend

{'slope_per_day': 0.05576608858134305,
 'annualized_change_per_year': 20.368563854335548,
 'start_value': 7.4366666666666665,
 'end_value': 15.385,
 'n_days': 200}

## Heatwave detection

In [9]:
heatwaves = detect_heatwaves(df)
print(f'{len(heatwaves)} heatwave events detected')
heatwaves.head(10)

20 heatwave events detected


,location,start,end,duration_days,max_temp,threshold
0,Almaty,2024-11-29,2024-12-01,3,16.90,14.600
1,Amsterdam,2024-12-01,2024-12-03,3,10.90,9.610
2,Berlin,2024-12-10,2024-12-12,3,13.60,11.305
3,Brussels,2024-12-09,2024-12-11,3,14.10,11.610
4,Cairo,2024-12-07,2024-12-09,3,23.30,21.310
5,Karachi,2024-11-16,2024-11-18,3,26.40,23.905
6,Lima,2024-08-10,2024-08-12,3,42.15,29.905
7,Lisbon,2024-12-05,2024-12-08,4,42.15,16.625
8,London,2024-12-11,2024-12-13,3,12.40,10.705
9,London,2024-12-15,2024-12-17,3,13.80,10.705


## Country temperature ranking

In [10]:
country_ranking(df, 'temperature_celsius').to_frame('avg_temp_c')

,avg_temp_c
country,
Kenya,29.78800
Singapore,28.94150
Malaysia,27.62225
Indonesia,27.08000
Colombia,26.87000
Nigeria,25.95625
Ethiopia,24.74525
Peru,23.76375
Thailand,22.36825
